[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/huggingface-nlp-certified/notebooks/day-07-trainer-api-finetune.ipynb#scrollTo=a1b2c3d4)

---
# Day 7 · Fine-Tuning BERT for Text Classification with the Trainer API
**certified-journeys / huggingface-nlp-certified** · Day 7 · Fine-Tuning

> **Goal for today:** Fine-tune a pre-trained BERT model on the SST-2 sentiment dataset using the Hugging Face Trainer API and save a production-ready checkpoint.


In [ ]:
%pip install -q transformers datasets evaluate accelerate scikit-learn


## Step 1 · The Trainer API at a glance

The Hugging Face `Trainer` abstracts the PyTorch training loop into four components:

| Component | What you provide |
|---|---|
| `model` | Any `PreTrainedModel` instance |
| `TrainingArguments` | Hyperparameters, output dir, evaluation strategy |
| `train_dataset` / `eval_dataset` | `datasets.Dataset` objects |
| `compute_metrics` | Function `(EvalPrediction) → dict` |

The Trainer handles: gradient accumulation, mixed-precision, distributed training, evaluation, checkpointing, and logging — all driven by `TrainingArguments`.

**Reference:** [https://huggingface.co/docs/transformers/training](https://huggingface.co/docs/transformers/training)


In [ ]:
# Core imports — brought in once, used throughout the notebook
import numpy as np
import evaluate
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)

# Load SST-2 — a binary sentiment classification dataset (positive / negative)
# SST-2 is part of the GLUE benchmark; labels: 0 = negative, 1 = positive
raw_datasets = load_dataset("glue", "sst2")
print(raw_datasets)
print("Sample train row:", raw_datasets["train"][0])


### What just happened?
- `load_dataset("glue", "sst2")` fetches ~67k training examples and a 872-example validation split.
- Each row has `sentence` (the text) and `label` (0 or 1).
- **The dataset is arrow-backed** — it lives on disk, not in RAM, so even large datasets don't OOM your notebook.
- The `idx` column is a row identifier from the original GLUE benchmark; we'll ignore it.


## Step 2 · Tokenize and set format for PyTorch

The Trainer expects tensors, not raw strings. We need to:
1. Tokenize every example with `AutoTokenizer`.
2. Call `dataset.set_format("torch")` so `__getitem__` returns tensors.

`DataCollatorWithPadding` handles dynamic padding at batch time — more efficient than padding the entire dataset to the global max length.


In [ ]:
MODEL_CHECKPOINT = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

def tokenize_function(examples):
    # truncation=True caps to the model's max length (512 for BERT)
    return tokenizer(examples["sentence"], truncation=True)

# batched=True processes ~1000 examples per call — much faster than row-by-row
tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)

# Remove columns the Trainer doesn't need: raw text and the GLUE row index
tokenized_datasets = tokenized_datasets.remove_columns(["sentence", "idx"])

# Rename 'label' → 'labels' — Trainer looks for the 'labels' key specifically
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")

# Tell the dataset to return PyTorch tensors on __getitem__
tokenized_datasets.set_format("torch")

# Verify shape of a single training example
sample = tokenized_datasets["train"][0]
print({k: v.shape for k, v in sample.items()})

# Dynamic padding collator — pads each batch to the longest sequence in that batch
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


### What just happened?
- `map(..., batched=True)` tokenized 67k examples in seconds using Arrow's columnar layout.
- After `set_format("torch")`, every column becomes a `torch.Tensor` — no manual `.to(torch.long)` needed.
- **`DataCollatorWithPadding` is the right choice** for variable-length text: padding at batch time avoids wasting compute on padding tokens that span the whole dataset's max length.
- The `labels` rename is required — `Trainer` computes the cross-entropy loss by looking for `batch["labels"]`.


## Step 3 · Define `compute_metrics` and initialize the model

`compute_metrics` receives an `EvalPrediction` object with:
- `.predictions` — raw logits, shape `(n_examples, num_labels)`
- `.label_ids` — ground truth integer labels

We use the `evaluate` library (Hugging Face) to compute accuracy. The function must return a `dict` of metric name → scalar.


In [ ]:
# Load the accuracy metric from the evaluate library
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # Take argmax over the class dimension to get predicted class indices
    predictions = np.argmax(logits, axis=-1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

# AutoModelForSequenceClassification adds a linear classification head on top of BERT
# num_labels=2 → binary sentiment (negative / positive)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT, num_labels=2
)

# The warning about 'some weights are newly initialized' is expected —
# the classification head doesn't exist in the pre-trained checkpoint
print(f"Model parameters: {model.num_parameters():,}")


### What just happened?
- `evaluate.load("accuracy")` pulls a metric object that works identically to sklearn but integrates with Trainer's evaluation loop.
- **`np.argmax(logits, axis=-1)`** converts raw logits to class indices — this is the equivalent of `softmax → argmax` but numerically more stable since we only need the argmax.
- `AutoModelForSequenceClassification.from_pretrained` loads BERT's 110M encoder weights and **randomly initializes** a 2-class linear head on top — that's the component we'll fine-tune.
- The "newly initialized" warning is not an error; it's confirming the head is fresh.


## Step 4 · Configure `TrainingArguments`

`TrainingArguments` is the single source of truth for the training run. Key parameters:

| Argument | Value | Why |
|---|---|---|
| `num_train_epochs` | 3 | Standard for BERT fine-tuning; more epochs risk overfitting |
| `per_device_train_batch_size` | 16 | Fits in Colab T4 VRAM with BERT-base |
| `evaluation_strategy` | `"epoch"` | Evaluate once per epoch so we can track progress |
| `save_strategy` | `"epoch"` | Save a checkpoint per epoch (required by `load_best_model_at_end`) |
| `load_best_model_at_end` | `True` | Automatically restore best checkpoint after training |
| `metric_for_best_model` | `"accuracy"` | Use accuracy (not loss) to pick the best checkpoint |


In [ ]:
training_args = TrainingArguments(
    output_dir="./bert-sst2-finetuned",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,   # larger eval batch — no gradient memory needed
    evaluation_strategy="epoch",
    save_strategy="epoch",           # must match evaluation_strategy for load_best_model_at_end
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,          # accuracy ↑ is better
    learning_rate=2e-5,              # standard BERT fine-tuning range: 1e-5 to 5e-5
    weight_decay=0.01,               # L2 regularization on non-bias parameters
    warmup_ratio=0.1,                # 10% of steps used for linear LR warmup
    logging_steps=50,
    report_to="none",                # disable wandb / tensorboard for this demo
)

print("Output dir:", training_args.output_dir)
print("Effective batch size:",
      training_args.per_device_train_batch_size * max(1, training_args.world_size))


### What just happened?
- `TrainingArguments` validates all hyperparameters at construction time — you get an error immediately if values are incompatible (e.g., `save_strategy != evaluation_strategy` when `load_best_model_at_end=True`).
- **`warmup_ratio=0.1`** implements a linear learning-rate schedule that ramps from 0 to `learning_rate` over 10% of total steps, then decays linearly to 0 — this prevents early catastrophic forgetting of the pre-trained weights.
- `report_to="none"` keeps the Colab output clean; in production set `"tensorboard"` or `"wandb"`.


## Step 5 · Instantiate `Trainer` and run training

The `Trainer` is the orchestration layer. It wires together the model, arguments, data, and metrics into a training loop. Calling `trainer.train()` starts the loop and prints per-epoch evaluation results.


In [ ]:
# Use a small subset for this demo so it runs quickly in Colab
# Remove the slicing to train on the full SST-2 dataset
small_train = tokenized_datasets["train"].select(range(2000))
small_eval  = tokenized_datasets["validation"].select(range(400))

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train,
    eval_dataset=small_eval,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# train() returns a TrainOutput with global_step, training_loss, and metrics
train_result = trainer.train()
print("\nTraining complete!")
print(f"  Global steps: {train_result.global_step}")
print(f"  Training loss: {train_result.training_loss:.4f}")


### What just happened?
- `Trainer.train()` ran 3 epochs with gradient updates, evaluation after each epoch, and checkpoint saving.
- Because `load_best_model_at_end=True`, the model weights in memory are now **the best checkpoint** — not necessarily the last epoch.
- **Eval loss and accuracy improve** across epochs as BERT adapts its representations to the SST-2 task.
- On the full dataset (~67k examples), expect ~92–93% validation accuracy after 3 epochs.


## Step 6 · Evaluate, save, and reload the fine-tuned model

After training we:
1. Run `trainer.evaluate()` for the final metrics report.
2. Save with `trainer.save_model()` — this writes `config.json`, `pytorch_model.bin`, and the tokenizer.
3. Reload with `AutoModelForSequenceClassification.from_pretrained()` to verify the saved checkpoint is usable.


In [ ]:
# Final evaluation on the validation set
eval_results = trainer.evaluate()
print("Final evaluation results:")
for k, v in eval_results.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")


In [ ]:
SAVE_DIR = "./bert-sst2-final"

# save_model writes model weights + config; also saves the tokenizer when
# tokenizer is passed to Trainer
trainer.save_model(SAVE_DIR)
print(f"Model saved to {SAVE_DIR}")

# Reload from disk — this is exactly what you'd do in a serving environment
reloaded_model = AutoModelForSequenceClassification.from_pretrained(SAVE_DIR)
reloaded_tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR)

print(f"Reloaded model class: {type(reloaded_model).__name__}")
print(f"Num labels: {reloaded_model.config.num_labels}")


In [ ]:
import torch

# Quick inference sanity check with the reloaded model
reloaded_model.eval()

test_sentences = [
    "This film was absolutely wonderful — a masterpiece.",
    "Terrible movie. I want my two hours back.",
    "The plot was mediocre but the acting saved it.",
]

label_names = ["NEGATIVE", "POSITIVE"]

with torch.no_grad():
    for sentence in test_sentences:
        inputs = reloaded_tokenizer(
            sentence, return_tensors="pt", truncation=True
        )
        logits = reloaded_model(**inputs).logits
        pred   = torch.argmax(logits, dim=-1).item()
        conf   = torch.softmax(logits, dim=-1)[0][pred].item()
        print(f"{label_names[pred]} ({conf:.2%}) — \"{sentence[:50]}...\"")


### What just happened?
- `trainer.save_model()` wrote a self-contained directory: model weights, tokenizer vocab, and config — everything `from_pretrained()` needs.
- **Reloading from disk** confirms the saved checkpoint is production-ready and portable — push this directory to the Hub with `model.push_to_hub("your-username/bert-sst2")` for public sharing.
- Manual inference with `torch.no_grad()` disables gradient tracking for faster, memory-efficient prediction.
- `torch.softmax` converts logits to probabilities; `torch.argmax` picks the winning class.


In [ ]:
# Challenge: Fine-tune for a different number of epochs and compare
# Your solution here
#
# 1. Create new TrainingArguments with num_train_epochs=1 and a new output_dir
# 2. Instantiate a fresh model from MODEL_CHECKPOINT (same num_labels=2)
# 3. Create a new Trainer with the same datasets and compute_metrics
# 4. Call trainer.train() and capture the eval_accuracy from trainer.evaluate()
# 5. Print a comparison table: epochs=1 vs epochs=3 accuracy
#
# Bonus: try learning_rate=5e-5 instead of 2e-5 — does it converge faster
# or overfit on the small subset?

# your_args = TrainingArguments(...)
# fresh_model = AutoModelForSequenceClassification.from_pretrained(...)
# ...


---
## Day 7 key concepts recap

| Concept | What to remember |
|---|---|
| `TrainingArguments` | Central config object — hyperparameters, strategy, and output live here |
| `evaluation_strategy='epoch'` | Runs full eval pass after each epoch; enables checkpoint selection |
| `load_best_model_at_end=True` | Requires `save_strategy == evaluation_strategy`; restores best weights |
| `dataset.set_format('torch')` | Converts Arrow columns to `torch.Tensor` without copying data |
| `DataCollatorWithPadding` | Pads each batch to the local max length — more efficient than global padding |
| `compute_metrics` | Receives raw logits + labels; must return `dict`; called at each eval step |
| `trainer.save_model()` | Writes weights + config + tokenizer; portable via `from_pretrained()` |

> **Tip:** Set `load_best_model_at_end=True` and `save_strategy='epoch'` in TrainingArguments so the Trainer automatically restores the checkpoint with the best eval metric.

---
## What's next
**Day 8** → Token Classification — NER with IOB Tagging and Sequence Labeling Heads: move from sentence-level prediction to token-level prediction and learn how subword tokenization complicates label alignment.

Mark Day 7 complete in your [tracker](../index.html).
